In [ ]:
from pathlib import Path
import sys

NOTEBOOK_DIR = Path.cwd()
PROJECT_ROOT = NOTEBOOK_DIR.parent if NOTEBOOK_DIR.name == "notebooks" else NOTEBOOK_DIR
SRC_DIR = PROJECT_ROOT / "src"

if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

print("Notebook:", NOTEBOOK_DIR)
print("Project :", PROJECT_ROOT)
print("Src     :", SRC_DIR)

# Prefer the CLI for reproducible runs:
# python scripts/train.py --epochs 30 --batch-size 32 --image-size 224

In [ ]:
import sys

print(sys.path[0])

In [ ]:
from pathlib import Path

import pandas as pd
from PIL import Image
from sklearn.metrics import classification_report, confusion_matrix

from data.labels import build_labels, copy_impaired_images
from data.dataset import MouseImageDataset as MouseDataset
from evaluation.experiment_modified import run_training_with_lr_schedule_early_stop_best_model
from paths import FIGURES_DIR, GRADCAM_DIR, IMAGES_MGS_DIR, IMAGES_PERFECT_DIR, MAIN_CSV, MGS_CSV, MODELS_DIR, MUZZLE_CROPS_DIR
from training.engine import build_dataloaders, build_transforms, run_training
from training.training import evaluate, evaluate_with_probs, make_group_split, train_one_epoch
from visualization.gradcam_functions import visualize_gradcam
from visualization.saliency_map import show_saliency, show_saliency_batch
from visualization.visualization_functions import visualize_binary_classifier_results


In [ ]:
IMG_DIR = IMAGES_PERFECT_DIR
print("Image directory:", IMG_DIR)
print("MGS CSV:", MGS_CSV)
print("Main CSV:", MAIN_CSV)

# 02 Image Classification

This notebook is an example workflow. Prefer the CLI scripts for reproducible runs:

```powershell
python scripts/train.py --epochs 30 --batch-size 32 --image-size 224
```

Use `IMAGES_MGS_DIR`, `IMAGES_PERFECT_DIR`, or `MUZZLE_CROPS_DIR` from `src/paths.py` to choose the image representation.

## select impared images & save (optional)

In [ ]:
df = build_labels(
    MGS_CSV,
    MAIN_CSV,
    IMG_DIR,
)

copy_impaired_images(
    df,
    PROJECT_ROOT / "outputs" / "image_impaired",
)

## sample visualization (option)

In [ ]:
print("Project root:", PROJECT_ROOT)
print("Image directory:", IMG_DIR)
print("MGS exists:", MGS_CSV.exists())
print("MAIN exists:", MAIN_CSV.exists())
print("Image folder exists:", IMG_DIR.exists())
print("Number of images:", len(list(IMG_DIR.glob("*.jpg"))) + len(list(IMG_DIR.glob("*.png"))))

print("Total usable images:", len(df))
print("Label distribution:")
print(df["label"].value_counts())

print("Subset distribution:")
print(df["subset"].value_counts())

df.head()


In [ ]:
# Optional: visualize random muzzle crops before training

sample_df = df.sample(n=min(25, len(df)), random_state=42)

plt.figure(figsize=(12, 12))

for i, (_, row) in enumerate(sample_df.iterrows()):
    img = Image.open(row["path"]).convert("RGB")
    plt.subplot(5, 5, i + 1)
    plt.imshow(img)
    plt.axis("off")
    plt.title(f'{row["index"]}label={row["label"]}', fontsize=8)

plt.tight_layout()
plt.show()

In [ ]:
print("Train size:", len(train_df))
print("Validation size:", len(val_df))

print("Train label distribution:")
print(train_df["label"].value_counts())

print("Validation label distribution:")
print(val_df["label"].value_counts())

In [ ]:
model, history = run_training_with_lr_schedule_early_stop_best_model(
    train_df=train_df,
    val_df=val_df,
    dataset_class=MouseDataset,
    train_one_epoch_fn=train_one_epoch,
    evaluate_fn=evaluate,
    save_path=MODELS_DIR / "mouse_wellbeing_convnext_tiny.pt",
    epochs=10,
    batch_size=32,
    lr=1e-4,
    weight_decay=1e-4,
)

# use lr schedule

In [ ]:
IMG_DIR = IMAGES_PERFECT_DIR
print("Using image directory:", IMG_DIR)

In [ ]:
print(NOTEBOOK_DIR.parent)

In [ ]:
df = build_labels(MGS_CSV, MAIN_CSV, IMG_DIR)
# We split by mouse identity, not by individual image.
# This avoids putting images from the same mouse in both train and validation.
train_df, val_df = make_group_split(df)

In [ ]:
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent

OUTPUT_DIR = PROJECT_ROOT / "outputs"
MODEL_DIR = OUTPUT_DIR / "models"

# 如果不存在就自动创建
MODEL_DIR.mkdir(parents=True, exist_ok=True)

save_path = MODEL_DIR / "the_best_muzzle_image_model_v1.pt"

In [ ]:
model, history = run_training_with_lr_schedule_early_stop_best_model(
    train_df=train_df,
    val_df=val_df,
    dataset_class=MouseDataset,
    train_one_epoch_fn=train_one_epoch,
    evaluate_fn=evaluate,
    save_path=save_path,
    epochs=1,
    batch_size=10,
    lr=1e-5,
    metric_name="macro_f1",
    scheduler_patience=2,
    early_stopping_patience=5
)

history_df = pd.DataFrame(history)

plt.figure(figsize=(6, 4))
plt.plot(history_df["epoch"], history_df["train_loss"], marker="o")
plt.xlabel("Epoch")
plt.ylabel("Train loss")
plt.title("Muzzle-only classifier training loss")
plt.grid(True)
plt.show()

## plotting

In [ ]:
train_loader, val_loader = build_dataloaders(
    train_df=train_df,
    val_df=val_df,
    dataset_class=MouseDataset,
    batch_size=32,
    image_size=224,
)

results = visualize_binary_classifier_results(
    model=model,
    val_loader=val_loader,
    evaluate_with_probs_fn=evaluate_with_probs,
    output_dir=FIGURES_DIR,
    prefix="muzzle_only",
    title_prefix="Muzzle Only",
)

## use grad cam saliency map

### 导入gradcam

In [ ]:
_, val_tfms = build_transforms()

visualize_gradcam(
    model=model,
    val_df=val_df,
    dataset_class=MouseDataset,
    target_layer=model.features[-1][-1].block[-1],
    output_dir=GRADCAM_DIR,
    num_images=5,
    transform=val_tfms,
)

### 手搓grad cam

In [ ]:
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
import numpy as np


class GradCAM:
    def __init__(self, model, target_layer):
        self.model = model
        self.target_layer = target_layer
        self.activations = None
        self.gradients = None

        self.forward_hook = target_layer.register_forward_hook(
            self.save_activation
        )
        self.backward_hook = target_layer.register_full_backward_hook(
            self.save_gradient
        )

    def save_activation(self, module, input, output):
        self.activations = output.detach()

    def save_gradient(self, module, grad_input, grad_output):
        self.gradients = grad_output[0].detach()

    def __call__(self, img_tensor, device, target_class=None):
        self.model.eval()

        img = img_tensor.unsqueeze(0).to(device)
        output = self.model(img)

        pred_class = output.argmax(dim=1).item()

        if target_class is None:
            target_class = pred_class

        score = output[0, target_class]

        self.model.zero_grad()
        score.backward()

        weights = self.gradients.mean(dim=(2, 3), keepdim=True)
        cam = (weights * self.activations).sum(dim=1)

        cam = F.relu(cam)
        cam = cam.squeeze().cpu().numpy()

        cam = (cam - cam.min()) / (cam.max() - cam.min() + 1e-8)

        return cam, pred_class, target_class

    def remove_hooks(self):
        self.forward_hook.remove()
        self.backward_hook.remove()


def show_gradcam(
    model,
    dataset,
    index,
    device,
    gradcam,
    target_class=None,
    class_names=None
):
    if class_names is None:
        class_names = ["well-being", "impaired"]

    img_tensor, true_label = dataset[index]

    cam, pred_class, used_class = gradcam(
        img_tensor=img_tensor,
        device=device,
        target_class=target_class
    )

    img = denormalize(img_tensor)

    cam_resized = F.interpolate(
        torch.tensor(cam).unsqueeze(0).unsqueeze(0),
        size=img.shape[:2],
        mode="bilinear",
        align_corners=False
    ).squeeze().numpy()

    plt.figure(figsize=(12, 4))

    plt.subplot(1, 3, 1)
    plt.imshow(img)
    plt.title(f"Original\nTrue: {class_names[int(true_label)]}")
    plt.axis("off")

    plt.subplot(1, 3, 2)
    plt.imshow(cam_resized, cmap="jet")
    plt.title(f"Grad-CAM\nClass: {class_names[used_class]}")
    plt.axis("off")

    plt.subplot(1, 3, 3)
    plt.imshow(img)
    plt.imshow(cam_resized, cmap="jet", alpha=0.45)
    plt.title(f"Overlay\nPred: {class_names[pred_class]}")
    plt.axis("off")

    plt.tight_layout()
    plt.show()

In [ ]:
# ==========================
# Build validation dataset
# ==========================
device = get_device()
_, val_tfms = build_transforms()

val_ds = MouseDataset(
    val_df,
    transform=val_tfms
)

# ==========================
# Create GradCAM object
# ==========================

target_layer = model.features[-1][-1].block[-1]

gradcam = GradCAM(
    model=model,
    target_layer=target_layer
)

number_of_images_2_show = 50 
for idx in range(number_of_images_2_show):
    show_gradcam(
        model=model,
        dataset=val_ds,
        index=idx,
        device=device,
        gradcam=gradcam,
        target_class=None
    )

# Fullimage

In [ ]:
IMG_DIR = IMAGES_MGS_DIR
print("Full-image directory:", IMG_DIR)

In [ ]:
df2 = build_labels(MGS_CSV, MAIN_CSV, IMG_DIR)
train_df2, val_df2 = make_group_split(df2)

In [ ]:
model2, history2 = run_training_with_lr_schedule_early_stop_best_model(
    train_df=train_df2,
    val_df=val_df2,
    dataset_class=MouseDataset,
    train_one_epoch_fn=train_one_epoch,
    evaluate_fn=evaluate,
    save_path="best_full_image_model.pt",
    epochs=5,
    batch_size=20,
    lr=1e-5,
    metric_name="macro_f1",
    scheduler_patience=2,
    early_stopping_patience=5
)

history_df = pd.DataFrame(history2)

plt.figure(figsize=(6, 4))
plt.plot(history_df["epoch"], history_df["train_loss"], marker="o")
plt.xlabel("Epoch")
plt.ylabel("Train loss")
plt.title("Muzzle-only classifier training loss")
plt.grid(True)
plt.show()

## plotting

In [ ]:
train_loader2, val_loader2 = build_dataloaders(
    train_df=train_df,
    val_df=val_df,
    dataset_class=MouseDataset,
    batch_size=32,
    image_size=224,
)

results = visualize_binary_classifier_results(
    model=model2,
    val_loader=val_loader2,
    evaluate_with_probs_fn=evaluate_with_probs,
    output_dir=FIGURES_DIR,
    prefix="full_image",
    title_prefix="full_image",
)

## grad cam

In [ ]:
_, val_tfms = build_transforms()

visualize_gradcam(
    model=model2,
    val_df=val_df2,
    dataset_class=MouseDataset,
    target_layer=model2.features[-1][-1].block[-1],
    output_dir=GRADCAM_DIR,
    num_images=5,
    transform=val_tfms,
)